# DoubleML PLR — Twitter Sentiment & Stock Returns
**Ashley Thompson — Capstone**

Doubly-robust ATE estimation via DoubleML Partially Linear Regression.
- **Run 1:** Average daily Twitter sentiment as continuous treatment
- **Run 2:** Negative tweet count as standalone treatment (Teti et al. 2019)

> Before running: go to **Runtime → Change runtime type → T4 GPU**

## 1. Install dependencies

In [ ]:
!pip install -q doubleml xgboost scikit-learn

## 2. Load data
**Option A — Google Drive (recommended for repeated runs)**  
Upload `panel_long.csv` to your Drive first, then run the Drive mount cell.

**Option B — Manual upload**  
Run the upload cell and select `panel_long.csv` from your computer.

Run whichever option you want, then skip the other.

In [ ]:
# Option A — Google Drive (data only)
from google.colab import drive
drive.mount('/content/drive')
DATA_PATH = '/content/drive/MyDrive/Colab Notebooks/Capstone/panel_long.csv'

In [ ]:
# Option B — Manual upload (data only)
from google.colab import files
uploaded = files.upload()
DATA_PATH = 'panel_long.csv'

In [ ]:
# GitHub output setup — run after whichever data option above
from google.colab import userdata
import os, subprocess

GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
REPO_DIR     = '/content/Capstone'
OUTPUT_PATH  = REPO_DIR + '/output/'

if os.path.exists(REPO_DIR):
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=True)
else:
    subprocess.run(
        ['git', 'clone',
         f'https://{GITHUB_TOKEN}@github.com/ATHOMP-03/Capstone.git',
         REPO_DIR],
        check=True,
    )
os.makedirs(OUTPUT_PATH, exist_ok=True)
print(f'Output path ready: {OUTPUT_PATH}')

## 3. Imports and setup

In [ ]:
import os
import numpy as np
import pandas as pd
import doubleml as dml
from xgboost import XGBRegressor

np.random.seed(42)
os.makedirs(OUTPUT_PATH, exist_ok=True)

long = pd.read_csv(DATA_PATH, parse_dates=['date'])
print(f"Loaded {len(long):,} rows x {long.shape[1]} cols")
long.head()

In [ ]:
# XGBoost learner — uses GPU automatically in Colab T4/A100 environments
# Change device='cpu' if you switch to a CPU runtime
def make_xgb():
    return XGBRegressor(
        n_estimators     = 500,
        learning_rate    = 0.05,
        max_depth        = 6,
        subsample        = 0.8,
        colsample_bytree = 0.8,
        device           = 'cuda',
        tree_method      = 'hist',
        random_state     = 42,
        n_jobs           = -1,
    )

## 4. Run 1 — Average daily Twitter sentiment
**Treatment:** `twitter_sent` (continuous, uniform [-1, 1])  
**Outcome:** `return` (open-to-close price change)  
**Confounders:** all variables except `px_open`, `px_close`, identifiers, outcome, and treatment

In [ ]:
confounders_1 = [
    'px_high', 'px_low', 'mkt_cap', 'total_equity', 'debt_to_equity',
    'volume', 'twitter_count', 'news_sent', 'rsi_30', 'ma_50',
    'twitter_neg_count', 'lag1', 'lag2', 'lag3', 'lag5', 'lag7',
]

df1 = (
    long[['return', 'twitter_sent'] + confounders_1]
    .dropna()
    .reset_index(drop=True)
)
print(f"N (complete cases): {len(df1):,}")

data_1 = dml.DoubleMLData(
    df1,
    y_col  = 'return',
    d_cols = 'twitter_sent',
    x_cols = confounders_1,
)

dml_1 = dml.DoubleMLPLR(
    data_1,
    ml_l    = make_xgb(),
    ml_m    = make_xgb(),
    n_folds = 5,
)
dml_1.fit()
print(dml_1.summary)

## 5. Run 2 — Negative tweet count (Teti et al. 2019)
**Treatment:** `twitter_neg_count` (count of negative tweets per day)  
Per Teti et al.: polarity-broken counts are significant predictors; total count is not. Using `twitter_neg_count` directly sidesteps neutral-tweet dilution in the Bloomberg index.  
**Outcome:** `return`  
**Confounders:** both `twitter_count` and `twitter_sent` stay in to control for total volume and overall sentiment direction independently.

In [ ]:
confounders_2 = [
    'px_high', 'px_low', 'mkt_cap', 'total_equity', 'debt_to_equity',
    'volume', 'twitter_sent', 'twitter_count', 'news_sent', 'rsi_30', 'ma_50',
    'lag1', 'lag2', 'lag3', 'lag5', 'lag7',
]

df2 = (
    long[['return', 'twitter_neg_count'] + confounders_2]
    .dropna()
    .reset_index(drop=True)
)
print(f"N (complete cases): {len(df2):,}")

data_2 = dml.DoubleMLData(
    df2,
    y_col  = 'return',
    d_cols = 'twitter_neg_count',
    x_cols = confounders_2,
)

dml_2 = dml.DoubleMLPLR(
    data_2,
    ml_l    = make_xgb(),
    ml_m    = make_xgb(),
    n_folds = 5,
)
dml_2.fit()
print(dml_2.summary)

## 6. Export — Publication-ready LaTeX table
Saves `dml_main_results.tex` to your output folder with both runs side by side.

In [ ]:
def dml_to_latex(models, labels, title, notes=''):
    """Build a publication-ready LaTeX table from DoubleMLPLR models."""
    def stars(p):
        if p < 0.01: return '***'
        if p < 0.05: return '**'
        if p < 0.1:  return '*'
        return ''

    all_treatments = []
    for m in models:
        for t in m.summary.index:
            if t not in all_treatments:
                all_treatments.append(t)

    ncols = len(models)
    col_fmt = 'l' + 'c' * ncols

    lines = [
        r'\begin{table}[htbp]',
        r'\centering',
        f'\\caption{{{title}}}',
        f'\\begin{{tabular}}{{{col_fmt}}}',
        r'\hline\hline',
        ' & ' + ' & '.join([f'({i+1}) {labels[i]}' for i in range(ncols)]) + r' \\',
        r'\hline',
    ]

    for t in all_treatments:
        t_label = t.replace('_', r'\_')
        coef_cells, se_cells = [], []
        for m in models:
            if t in m.summary.index:
                coef = m.summary.loc[t, 'coef']
                se   = m.summary.loc[t, 'std err']
                pval = m.summary.loc[t, 'P>|t|']
                coef_cells.append(f'{coef:.6f}{stars(pval)}')
                se_cells.append(f'({se:.6f})')
            else:
                coef_cells.append('---')
                se_cells.append('')
        lines.append(t_label + ' & ' + ' & '.join(coef_cells) + r' \\')
        lines.append('& ' + ' & '.join(se_cells) + r' \\')

    lines += [
        r'\hline',
        'Estimator & ' + ' & '.join(['DoubleML PLR'] * ncols) + r' \\',
        'ML Learner & ' + ' & '.join(['XGBoost (GPU)'] * ncols) + r' \\',
        r'\hline\hline',
    ]
    if notes:
        lines.append(
            f'\\multicolumn{{{ncols + 1}}}{{l}}'
            f'{{\\footnotesize \\textit{{Notes:}} {notes}}} \\\\'
        )
    lines += [r'\end{tabular}', r'\end{table}']
    return '\n'.join(lines)

In [ ]:
tex = dml_to_latex(
    [dml_1, dml_2],
    ['Twitter Sentiment', 'Neg Tweet Count'],
    title='DoubleML PLR --- Twitter Sentiment and Stock Returns',
    notes=(
        'Heteroskedasticity-robust SE in parentheses. '
        'XGBoost learner with 5-fold cross-fitting, 20 repetitions, 1000 bootstrap draws. '
        'Run 2 per Teti et al. (2019): polarity-broken tweet counts outperform total count. '
        '*** p$<$0.01, ** p$<$0.05, * p$<$0.1.'
    ),
)
with open(OUTPUT_PATH + 'dml_main_results.tex', 'w') as f:
    f.write(tex)
print('Saved dml_main_results.tex')
print(tex)

In [ ]:
# Push outputs to GitHub
import os, subprocess

os.chdir(REPO_DIR)
subprocess.run(['git', 'config', 'user.email', 'ATHOMP-03@users.noreply.github.com'])
subprocess.run(['git', 'config', 'user.name',  'ATHOMP-03'])
subprocess.run(['git', 'add', 'output/'])
result = subprocess.run(['git', 'diff', '--cached', '--quiet'])
if result.returncode != 0:
    subprocess.run(['git', 'commit', '-m', 'Add DoubleML main results from Colab'], check=True)
    subprocess.run(['git', 'push'], check=True)
    print('Pushed to GitHub.')
else:
    print('Nothing new to commit.')